In [ ]:
# pip install azure-storage-blob pandas

import pandas as pd
from io import BytesIO
from azure.storage.blob import BlobServiceClient

account_name = "insurancedataset2026"
account_key = "YOUR_ACCOUNT_KEY"

connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={account_name};"
    f"AccountKey={account_key};"
    f"EndpointSuffix=core.windows.net"
)

# Connect to Azure Blob Storage
blob_service = BlobServiceClient.from_connection_string(connection_string)

# Source container
source_container = "source"

source_client = blob_service.get_container_client(source_container)

# Read Home Credit application_train.csv
blob_client = source_client.get_blob_client("application_train.csv")

data = blob_client.download_blob().readall()

df = pd.read_csv(BytesIO(data))

# Print rows and columns
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

# Print column names
print("Column Names:", df.columns.tolist())

# Display first 5 records
print(df.head())

In [ ]:
# pip install azure-storage-blob pandas

import pandas as pd
from io import BytesIO
from azure.storage.blob import BlobServiceClient

account_name = "insurancedataset2026"
account_key = "YOUR_ACCOUNT_KEY"

connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={account_name};"
    f"AccountKey={account_key};"
    f"EndpointSuffix=core.windows.net"
)

# Connect to Azure Blob Storage
blob_service = BlobServiceClient.from_connection_string(connection_string)

# Source container
source_container = "source"

source_client = blob_service.get_container_client(source_container)

# Read Home Credit application_train.csv
blob_client = source_client.get_blob_client("application_train.csv")

data = blob_client.download_blob().readall()

df = pd.read_csv(BytesIO(data))

# Print rows and columns
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

# Print column names
print("Column Names:", df.columns.tolist())

# Display first 5 records
print(df.head())

In [ ]:
# pip install azure-storage-blob pandas

import os
import io
import uuid
import hashlib
import pandas as pd

from datetime import datetime, timezone
from azure.storage.blob import BlobServiceClient

# ============================================================
# 1. AZURE STORAGE CONFIGURATION
# ============================================================

account_name = "insurancedataset2026"
account_key = "YOUR_ACCOUNT_KEY"

source_container = "source"
destination_container = "destination"
source_file = "application_train.csv"

connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={account_name};"
    f"AccountKey={account_key};"
    f"EndpointSuffix=core.windows.net"
)

blob_service = BlobServiceClient.from_connection_string(
    connection_string
)

source_client = blob_service.get_container_client(
    source_container
)

destination_client = blob_service.get_container_client(
    destination_container
)

# ============================================================
# 2. INITIALIZE VALIDATION LOG TABLE
# ============================================================

run_id = str(uuid.uuid4())
run_time = datetime.now(timezone.utc)

validation_logs = []

def log_validation(check_name, status, total, failed, description):

    validation_logs.append({
        "run_id": run_id,
        "run_timestamp": run_time.isoformat(),
        "source_file": source_file,
        "validation_name": check_name,
        "status": status,
        "total_records": int(total),
        "failed_records": int(failed),
        "passed_records": int(total - failed),
        "description": description
    })

    print(f"{check_name}: {status} | Failed: {failed}")


# ============================================================
# 3. READ HOME CREDIT CSV FROM AZURE BLOB
# ============================================================

blob_client = source_client.get_blob_client(source_file)

file_bytes = blob_client.download_blob().readall()

file_size = len(file_bytes)
file_hash = hashlib.sha256(file_bytes).hexdigest()

df = pd.read_csv(
    io.BytesIO(file_bytes),
    low_memory=False
)

total_rows = len(df)
total_columns = len(df.columns)

print("Rows:", total_rows)
print("Columns:", total_columns)
print("File Size:", file_size)
print("SHA256:", file_hash)


# ============================================================
# 4. FILE VALIDATION
# ============================================================

log_validation(
    "FILE_NOT_EMPTY",
    "PASS" if file_size > 0 else "FAIL",
    1,
    int(file_size == 0),
    "Check whether source file is empty"
)

log_validation(
    "DATASET_NOT_EMPTY",
    "PASS" if total_rows > 0 else "FAIL",
    1,
    int(total_rows == 0),
    "Check whether dataset contains records"
)


# ============================================================
# 5. COLUMN COUNT VALIDATION
# ============================================================

expected_columns = 122

failed = int(total_columns != expected_columns)

log_validation(
    "COLUMN_COUNT_CHECK",
    "PASS" if failed == 0 else "FAIL",
    1,
    failed,
    f"Expected 122 columns, found {total_columns}"
)


# ============================================================
# 6. REQUIRED COLUMN VALIDATION
# ============================================================

required_columns = [
    "SK_ID_CURR",
    "TARGET",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

log_validation(
    "REQUIRED_COLUMNS_CHECK",
    "PASS" if not missing_columns else "FAIL",
    len(required_columns),
    len(missing_columns),
    f"Missing columns: {missing_columns}"
)


# ============================================================
# 7. DUPLICATE COLUMN VALIDATION
# ============================================================

duplicate_columns = int(df.columns.duplicated().sum())

log_validation(
    "DUPLICATE_COLUMN_CHECK",
    "PASS" if duplicate_columns == 0 else "FAIL",
    total_columns,
    duplicate_columns,
    "Check duplicate column names"
)


# ============================================================
# 8. NULL VALUE VALIDATION - ALL COLUMNS
# ============================================================

for column in df.columns:

    failed = int(df[column].isnull().sum())

    log_validation(
        f"NULL_CHECK_{column}",
        "PASS" if failed == 0 else "WARN",
        total_rows,
        failed,
        f"Null values in {column}"
    )


# ============================================================
# 9. CRITICAL COLUMN NULL VALIDATION
# ============================================================

critical_columns = [
    "SK_ID_CURR",
    "TARGET",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "DAYS_BIRTH"
]

for column in critical_columns:

    if column not in df.columns:
        continue

    failed = int(df[column].isnull().sum())

    log_validation(
        f"CRITICAL_NULL_{column}",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        f"Critical field {column} cannot be null"
    )


# ============================================================
# 10. DUPLICATE RECORD VALIDATION
# ============================================================

duplicate_rows = int(df.duplicated().sum())

log_validation(
    "DUPLICATE_RECORD_CHECK",
    "PASS" if duplicate_rows == 0 else "FAIL",
    total_rows,
    duplicate_rows,
    "Check complete duplicate records"
)


# ============================================================
# 11. PRIMARY KEY UNIQUENESS VALIDATION
# ============================================================

if "SK_ID_CURR" in df.columns:

    duplicate_ids = int(
        df["SK_ID_CURR"].duplicated().sum()
    )

    log_validation(
        "PRIMARY_KEY_UNIQUENESS",
        "PASS" if duplicate_ids == 0 else "FAIL",
        total_rows,
        duplicate_ids,
        "SK_ID_CURR must be unique"
    )


# ============================================================
# 12. TARGET VALIDATION
# ============================================================

if "TARGET" in df.columns:

    invalid_target = (
        df["TARGET"].notna()
        & ~df["TARGET"].isin([0, 1])
    )

    failed = int(invalid_target.sum())

    log_validation(
        "TARGET_DOMAIN_CHECK",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "TARGET must contain only 0 or 1"
    )


# ============================================================
# 13. NUMERIC DATA TYPE VALIDATION
# ============================================================

numeric_columns = [
    "SK_ID_CURR",
    "TARGET",
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

for column in numeric_columns:

    if column not in df.columns:
        continue

    converted = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    invalid = (
        df[column].notna()
        & converted.isna()
    )

    failed = int(invalid.sum())

    log_validation(
        f"NUMERIC_TYPE_{column}",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        f"Validate numeric data type for {column}"
    )


# ============================================================
# 14. NEGATIVE AMOUNT VALIDATION
# ============================================================

amount_columns = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE"
]

for column in amount_columns:

    if column not in df.columns:
        continue

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    failed = int((values < 0).sum())

    log_validation(
        f"NEGATIVE_AMOUNT_{column}",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        f"Check negative values in {column}"
    )


# ============================================================
# 15. AGE VALIDATION
# ============================================================

if "DAYS_BIRTH" in df.columns:

    days_birth = pd.to_numeric(
        df["DAYS_BIRTH"],
        errors="coerce"
    )

    age = -days_birth / 365.25

    invalid_age = (
        (age < 18)
        | (age > 100)
        | (days_birth >= 0)
    )

    failed = int(invalid_age.sum())

    log_validation(
        "AGE_RANGE_CHECK",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "Age must be between 18 and 100 years"
    )


# ============================================================
# 16. EMPLOYMENT DAYS VALIDATION
# ============================================================

if "DAYS_EMPLOYED" in df.columns:

    employment = pd.to_numeric(
        df["DAYS_EMPLOYED"],
        errors="coerce"
    )

    sentinel = employment == 365243

    log_validation(
        "EMPLOYMENT_SENTINEL_CHECK",
        "WARN" if sentinel.sum() > 0 else "PASS",
        total_rows,
        int(sentinel.sum()),
        "365243 is a known special employment value"
    )

    invalid_employment = (
        (employment > 0)
        & (~sentinel)
    )

    failed = int(invalid_employment.sum())

    log_validation(
        "EMPLOYMENT_DAYS_CHECK",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "Check invalid positive employment days"
    )


# ============================================================
# 17. CHILDREN COUNT VALIDATION
# ============================================================

if "CNT_CHILDREN" in df.columns:

    children = pd.to_numeric(
        df["CNT_CHILDREN"],
        errors="coerce"
    )

    failed = int((children < 0).sum())

    log_validation(
        "CHILDREN_COUNT_CHECK",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "Children count cannot be negative"
    )


# ============================================================
# 18. EXTERNAL SOURCE SCORE VALIDATION
# ============================================================

score_columns = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

for column in score_columns:

    if column not in df.columns:
        continue

    scores = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    invalid = (
        (scores < 0)
        | (scores > 1)
    )

    failed = int(invalid.sum())

    log_validation(
        f"SCORE_RANGE_{column}",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "External score must be between 0 and 1"
    )


# ============================================================
# 19. CATEGORICAL DOMAIN VALIDATION
# ============================================================

category_rules = {
    "NAME_CONTRACT_TYPE": [
        "Cash loans",
        "Revolving loans"
    ],
    "CODE_GENDER": [
        "M",
        "F",
        "XNA"
    ],
    "FLAG_OWN_CAR": [
        "Y",
        "N"
    ],
    "FLAG_OWN_REALTY": [
        "Y",
        "N"
    ]
}

for column, valid_values in category_rules.items():

    if column not in df.columns:
        continue

    invalid = (
        df[column].notna()
        & ~df[column].isin(valid_values)
    )

    failed = int(invalid.sum())

    log_validation(
        f"CATEGORY_DOMAIN_{column}",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        f"Allowed values: {valid_values}"
    )


# ============================================================
# 20. ZERO CREDIT VALIDATION
# ============================================================

if "AMT_CREDIT" in df.columns:

    credit = pd.to_numeric(
        df["AMT_CREDIT"],
        errors="coerce"
    )

    failed = int((credit == 0).sum())

    log_validation(
        "ZERO_CREDIT_CHECK",
        "PASS" if failed == 0 else "FAIL",
        total_rows,
        failed,
        "Credit amount must be greater than zero"
    )


# ============================================================
# DISPLAY VALIDATION LOG TABLE
# ============================================================

validation_log_df = pd.DataFrame(validation_logs)

print("\n========== BRONZE VALIDATION LOG TABLE ==========")

print(validation_log_df.to_string(index=False))

print("\n========== VALIDATION SUMMARY ==========")

print("Total Rows:", total_rows)
print("Total Columns:", total_columns)
print("Total Validation Checks:", len(validation_log_df))

print(
    "Passed Checks:",
    (validation_log_df["status"] == "PASS").sum()
)

print(
    "Failed Checks:",
    (validation_log_df["status"] == "FAIL").sum()
)

print(
    "Warning Checks:",
    (validation_log_df["status"] == "WARN").sum()
)